Consistency check: same image, same prompt, 5 times.

Pins: the RQ1 human pins (minus `excluded_sources`), both tasks, all three models, repeats 0-4.
Repeat 0 shares the main run's cache key, so it is paid only once whichever runs first.
Paid: 195 × 4 × 3 × 2 = 4,680 new calls (~$19); dry run must show 4,680, ~5,850 = repeat 0 misses the main-run cache. Needs `render_main.ipynb` first.
Metrics are computed in `scoring.ipynb` (repeat 0 from the main-run answers).

In [ ]:
import duckdb
import pandas as pd

from thesis.config import REPO_ROOT, load_config
from thesis.vlm.run import run_dataset

out_dir = REPO_ROOT / "data" / "dataset_v2"
REPEATS = range(5)
prompts = load_config("prompts")
TASK_PROMPTS = {
    "verification": next(p["text"] for p in prompts["prompts"] if p["id"] == "roleplaying"),
    "distance": next(p["text"] for p in prompts["distance_prompts"] if p["id"] == "roleplaying"),
}

excluded = {entry["source_id"] for entry in load_config("scoring")["excluded_sources"]}
human = pd.read_json(out_dir / "gold" / "human_on_roof.jsonl", lines=True)
human = human.loc[~human["source_id"].isin(excluded)]
with duckdb.connect(str(out_dir / "manifest.duckdb"), read_only=True) as connection:
    renders = connection.execute("SELECT pin_id, image_path FROM renders WHERE preset_id = 'google_pin_red'").fetch_df()
consistency_pins = human[["pin_id"]].merge(renders, on="pin_id")
images = [(out_dir / path).read_bytes() for path in consistency_pins["image_path"]]
len(consistency_pins)

195

In [ ]:
# dry run: count cache misses, no requests sent
from thesis.vlm.run import configured_clients
from thesis.vlm.vlm_adapter import _cache_key, _cache_path, _prepare_request

misses = {}
for client in configured_clients():
    settings = client._request_settings()
    for task, prompt in TASK_PROMPTS.items():
        for repeat in REPEATS:
            misses[(client.provider, task, repeat)] = sum(
                not _cache_path(
                    client.cache_dir,
                    _cache_key(settings, (r := _prepare_request(image, prompt)).image_sha256, r.prompt_sha256, repeat),
                ).exists()
                for image in images
            )
misses = pd.Series(misses)
misses.unstack(), misses.sum()

(                        0  1  2  3  4
 anthropic distance      0  0  0  0  0
           verification  0  0  0  0  0
 google    distance      0  0  0  0  0
           verification  0  0  0  0  0
 openai    distance      0  0  0  0  0
           verification  0  0  0  0  0,
 np.int64(0))

In [ ]:
# paid run
rows = []
for task, prompt in TASK_PROMPTS.items():
    for repeat in REPEATS:
        results = run_dataset([(image, prompt) for image in images], task=task, repeat=repeat)
        for provider, responses in results.items():
            for pin_id, response in zip(consistency_pins["pin_id"], responses):
                rows.append(
                    {
                        "task": task,
                        "repeat": repeat,
                        "provider": provider,
                        "pin_id": pin_id,
                        "parsed_answer": response.vlm_response.parsed_answer,
                        "raw_text": response.vlm_response.raw_text,
                        "refusal_error_state": response.vlm_response.refusal_error_state,
                    }
                )
answers = pd.DataFrame(rows)
answers.to_parquet(out_dir / "consistency_answers.parquet")
answers.groupby(["task", "provider"]).size()

task          provider 
distance      anthropic    975
              google       975
              openai       975
verification  anthropic    975
              google       975
              openai       975
dtype: int64